# 02 · Functions & Scope — **Depth**

> **Pull this notebook when:** you're building anything that returns a configured function — samplers,
> the mood-update step, swappable components. Closures and scope are the machinery under all of it, and
> the foundation for the Decorators notebook.

Predict before running; answers hidden. (This is the template you already saw, now closing with a
cumulative milestone.)

---

## 2.1 — Predict: what do these return?

```python
funcs = [lambda: i for i in range(3)]
results = [f() for f in funcs]
```

In [ ]:
funcs = [lambda: i for i in range(3)]
results = [f() for f in funcs]
print(results)

<details>
<summary>▶ Reveal</summary>

`[2, 2, 2]`. A closure captures the **variable**, not its value at creation. All three lambdas share
the same `i`; the loop ends with `i == 2`, and only then are they called. Fix by freezing the value at
definition with a default arg: `lambda i=i: i`. This bites whenever you build configured functions in
a loop — exactly what a registry does.

</details>

## 2.2 — Modify: make `results == [0, 1, 2]`

Fix the closure so each function remembers its own `i`. Don't restructure.

In [ ]:
funcs = [lambda: i for i in range(3)]
results = [f() for f in funcs]

# Tests
assert results == [0, 1, 2], f"got {results}"
print("2.2 passed")

## 2.3 — Predict: run, print, or raise?

```python
x = "global"
def reads(): return x
def reads_then_assigns():
    print(x)
    x = "local"
    return x
```
What does `reads()` do? What does `reads_then_assigns()` do?

In [ ]:
x = "global"
def reads():
    return x
def reads_then_assigns():
    print(x)
    x = "local"
    return x
print(reads())
print(reads_then_assigns())

<details>
<summary>▶ Reveal</summary>

`reads()` returns `"global"`. `reads_then_assigns()` raises **`UnboundLocalError`** on `print(x)`.
Python decides a name's scope for the *whole* function body at compile time: because `x` is assigned
*somewhere* in the body, it's local *everywhere* in it — including the earlier read, where it's not yet
bound. Assignment anywhere makes the name local everywhere. To read-then-rebind an outer name you must
declare `global`/`nonlocal`.

</details>

## 2.4 — Modify: make the counter count

Add one line so it returns 1, 2, 3.

In [ ]:
def make_counter():
    count = 0
    def increment():
        count = count + 1
        return count
    return increment
c = make_counter()

# Tests
assert c() == 1 and c() == 2 and c() == 3
print("2.4 passed")

<details>
<summary>▶ Why</summary>

`nonlocal count` as the first line. Without it, `count = count + 1` makes `count` local to `increment`
(the 2.3 rule), so the read fails or resets each call. `nonlocal` rebinds the enclosing `count`;
`global` does the same for module-level names.

</details>

## 2.5 — Build: a closure that remembers a running average

`make_averager()` returns a function; each call adds a number and returns the average so far. Two
averagers must be independent. Annotate with `Callable`.

In [ ]:
from typing import Callable
def make_averager() -> Callable[[float], float]:
    # YOUR CODE HERE
    pass

# Tests
avg = make_averager()
assert avg(10) == 10.0 and avg(20) == 15.0 and avg(30) == 20.0
assert make_averager()(100) == 100.0
print("2.5 passed")

<details>
<summary>▶ Note</summary>

You didn't need `nonlocal` — you *mutate* `history` (`.append`), never *rebind* it. `nonlocal` is for
rebinding; mutating an object the closure already points at needs no declaration. Rebind vs mutate is
the same distinction from the aliasing notebook.

</details>

## 2.6 — Three ways to carry state (and when each)

Implement a running-sum accumulator as **(1)** a closure, **(2)** a class with `__call__`, **(3)** the
mutable-default hack. All correct; the point is the trade-offs (see the answer).

In [ ]:
from typing import Callable
def acc_closure() -> Callable[[int], int]:
    # YOUR CODE HERE
    pass
class AccClass:
    def __init__(self):
        # YOUR CODE HERE
        pass
    def __call__(self, x: int) -> int:
        # YOUR CODE HERE
        pass
def acc_hack(x: int, _total=None) -> int:
    # the hack: use a mutable default to persist state (implement to understand, never ship)
    # YOUR CODE HERE
    pass

# Tests
a1 = acc_closure()
assert a1(10) == 10 and a1(5) == 15
a2 = AccClass()
assert a2(10) == 10 and a2(5) == 15
assert acc_hack(10) == 10 and acc_hack(5) == 15
print("2.6 passed")

<details>
<summary>▶ When each</summary>

**Closure** — default for a small stateful function; state is truly private, nothing to instantiate.
**`__call__` class** — when state must be inspected, reset, or extended (`a2.total` is readable; add a
`.reset()`). This is what a sampler or mood model wants. **Mutable-default hack** — essentially never;
it exploits the trap, shares one accumulator across all callers, and reads as a bug. Know it to remove
it. The recurring question: *does anything outside need to see or change the state?* If yes, object; if
no, closure.

</details>

## 2.7 — Written spec: configurable formatter, keyword-only

Write `make_formatter` returning a function that takes a string and applies options fixed at creation:
`upper` (bool) and `prefix` (str). Make `upper`/`prefix` **keyword-only** (a bare `*` before them) so
callers must name them.

In [ ]:
from typing import Callable
def make_formatter(*, upper: bool = False, prefix: str = "") -> Callable[[str], str]:
    # YOUR CODE HERE
    pass

# Tests
shout = make_formatter(upper=True, prefix=">> ")
assert shout("hello") == ">> HELLO"
assert make_formatter()("hello") == "hello"
raised = False
try:
    make_formatter(True)
except TypeError:
    raised = True
assert raised
print("2.7 passed")

<details>
<summary>▶ Why keyword-only</summary>

The bare `*` makes everything after it keyword-only, so `make_formatter(True)` raises `TypeError`.
`make_formatter(upper=True, prefix=">> ")` documents itself at the call site; positional flags
(`make_formatter(True, ">> ")`) don't. For functions with several option flags — which Mara's persona
and sampling config will have — keyword-only arguments are a standard readability move.

</details>

## ★ Milestone 02 — A transform registry with composition

Synthesize closures + first-class functions into the pattern you'll use for swappable components in
Act 1. Build `make_pipeline()` returning **three** functions: `register(name)` (a decorator-style
registrar that stores a function and returns it unchanged), `get(name)` (retrieve by name), and
`run(names, value)` (apply the named transforms left-to-right to `value`, feeding each result into the
next). The registry dict is private to the closure.

This is closures, first-class functions, and composition doing real work — and the literal shape of
config-driven component selection later in the project.

(build it below)

In [ ]:
def make_pipeline():
    # YOUR CODE HERE
    # private dict; register(name) -> decorator that stores+returns func;
    # get(name) -> stored func; run(names, value) -> apply each named transform in order
    pass

# Tests
register, get, run = make_pipeline()

@register("double")
def double(x): return x * 2

@register("inc")
def inc(x): return x + 1

@register("neg")
def neg(x): return -x

assert get("double")(5) == 10
assert run(["double", "inc"], 5) == 11        # (5*2)+1
assert run(["inc", "double"], 5) == 12        # (5+1)*2  — order matters
assert run(["double", "inc", "neg"], 3) == -7 # ((3*2)+1) negated
assert run([], 42) == 42                      # no transforms → unchanged
# register returns the function unchanged, still callable directly:
assert double(4) == 8
print("Milestone 02 passed")

<details>
<summary>▶ The through-line</summary>

Three nested-scope functions sharing one private `table`. `register` is a decorator factory (returns
`deco`, which stores and returns the function unchanged). `run` *composes* transforms by threading the
value through each in order — so `run(["double","inc"], 5)` is `inc(double(5))`. Order dependence is
the whole point of a pipeline. In Weeks 4–5 this is exactly how `config["attention"]` will pick which
registered component runs.

</details>